In [42]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

from langgraph.graph.message import BaseMessage, add_messages
from typing import Annotated, TypedDict

from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

from langchain_core.output_parsers import StrOutputParser

import sys

load_dotenv()


True

In [43]:
chatModel = ChatOpenAI(model='gpt-4o')
MCP_CONFIG = {
    "math": {
        "transport": "stdio",
        "command": sys.executable,   # the venv python running this notebook
        "args": ["/Users/sriharshamadireddy/Desktop/Personal/Learning/DataScience/genai/langgraph/14. Langgraph with MCP/practice/mcpServer.py"],
    }
}

mcp_client = MultiServerMCPClient(MCP_CONFIG)

tools = await mcp_client.get_tools()

print(tools)

chatModel = chatModel.bind_tools(tools)
toolNode = ToolNode(tools)

[StructuredTool(name='calculator', description='performs necessary operation', args_schema={'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}, 'c': {'type': 'string'}}, 'required': ['a', 'b', 'c'], 'type': 'object', 'additionalProperties': False}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x13c26aac0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x16192cd60>)]


In [44]:
class State(TypedDict):
    messages: Annotated[list[str], add_messages]

In [59]:
stroutputParser = StrOutputParser()

def chat(state: State) -> State:
    response = chatModel.invoke(state['messages'])
    return {"messages": [response]}


In [60]:
graph = StateGraph(State)
graph.add_node('chat', chat)
graph.add_node('tools',  toolNode)
graph.add_edge(START, 'chat')
graph.add_conditional_edges('chat', tools_condition)
graph.add_edge('tools', 'chat')


workflow = graph.compile()

In [61]:
await workflow.ainvoke({"messages": [HumanMessage("what is 1 + 2")]})

{'messages': [HumanMessage(content='what is 1 + 2', additional_kwargs={}, response_metadata={}, id='6a9048bb-72ee-4364-90a6-5fdc69f9ffca'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 55, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EaHMafgZNng7SeAxFp1KYyWFyHN7H', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a14e50-7b32-75d2-9250-d1b126e784d3-0', tool_calls=[{'name': 'calculator', 'args': {'a': 1, 'b': 2, 'c': '+'}, 'id': 'call_i2lnrqJSaUYbMyVRvYYZJHbR', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 22,